# 13 — UMAP + HDBSCAN patient stratification and burden heatmaps

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis.lib.data import load_label

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA     = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA"
LACUNA_DIR = LACUNA / "derivatives"
EXP_DIR    = PROJECT_ROOT / "outputs" / "subject_level_analyses"
FIG_DIR    = EXP_DIR / "figures"
EXP_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

TARGETS = [
    "cognition-PSCI_present",
    "cognition-aef_domain_score",
    "cognition-ps_domain_score",
    "cognition-language_domain_score",
    "cognition-verbalmemory_domain_score",
    "cognition-vsfunctions_domain_score",
    "cognition-vsmemory_domain_score",
]

In [ ]:
# 1. UMAP + HDBSCAN per view (sLNM headline + lesion/fLNM supplements)
import umap
import hdbscan

VIEW_TO_LOAD = {
    "sLNM":   ("snm", "disconnectionpct"),
    "lesion": ("rd",  "damagepct"),
    "flnm":   ("fnm", "tmap"),
}


def cluster_view_umap_hdbscan(view):
    out_csv = EXP_DIR / f"umap_{view}_subjects.csv"
    if out_csv.exists():
        print(f"[skip] {out_csv.name} exists")
        return
    method, desc = VIEW_TO_LOAD[view]
    clinical = pd.read_csv(CLIN_PATH, index_col=0)
    feat = load_label(LACUNA_DIR, "acuteinfarct", method, desc)
    common = clinical.index.intersection(feat.index)
    clinical = clinical.loc[common]
    X = np.nan_to_num(feat.loc[common].values)
    print(f"[{view}] N = {len(common)}, {X.shape[1]} parcels")

    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=2,
                        metric="euclidean", random_state=42)
    emb = reducer.fit_transform(X)
    cl = hdbscan.HDBSCAN(min_cluster_size=50, min_samples=10,
                         cluster_selection_method="eom")
    labels = cl.fit_predict(emb)
    print(f"[{view}] HDBSCAN: {int(labels.max() + 1)} clusters, "
          f"{int((labels == -1).sum())} noise")

    out = pd.DataFrame({"subject_id": common, "umap_x": emb[:, 0],
                        "umap_y": emb[:, 1], "cluster": labels})
    out.to_csv(out_csv, index=False)

    stats = []
    for c in sorted(out["cluster"].unique()):
        sub_ids = out.loc[out["cluster"] == c, "subject_id"].values
        sub_clin = clinical.loc[sub_ids]
        row = {"cluster": int(c), "n": len(sub_ids)}
        for t in TARGETS:
            y = sub_clin[t].dropna()
            row[f"{t}_n"] = int(len(y))
            row[f"{t}_rate"] = float(y.mean()) if len(y) else np.nan
        stats.append(row)
    pd.DataFrame(stats).to_csv(EXP_DIR / f"umap_{view}_cluster_stats.csv", index=False)
    print(f"[{view}] saved {out_csv.name} + cluster stats")


for view in VIEW_TO_LOAD:
    cluster_view_umap_hdbscan(view)

In [ ]:
# 2. Per-cluster mean sLNM maps + axial slices (Fig 4b input)
import nibabel as nib
from nilearn import plotting

CELLS_SLNM = PROJECT_ROOT / "outputs" / "feature_importance" / "figures" / "cells_fig2_clusters"
CELLS_SLNM.mkdir(parents=True, exist_ok=True)
N_CUTS_PER_CLUSTER = 3

# Hand-picked z-cuts (MNI mm) per 0-based HDBSCAN cluster id; others auto-pick.
CLUSTER_Z_CUTS = {
    0: (-15, -5,  5),    # 1: L PCA / AChA
    1: (10,  20, 30),    # 2: ACA
    2: (-35, -25, -15),  # 3: Infratentorial
    3: (0,   10, 20),    # 4: L deep subcortical
    4: (15,  25, 35),    # 5: L MCA
    5: (0,   10, 20),    # 6: R deep subcortical
    6: (15,  25, 35),    # 7: R MCA
    7: (-15, -5,  5),    # 8: R PCA / AChA
    8: (-50, -35, -20),  # 9: Diverse
}


def _slnm_nii_path(sid):
    return (LACUNA / f"sub-{sid}" / "ses-01" / "anat" /
            f"sub-{sid}_ses-01_label-acuteinfarct_method-snm_"
            "space-MNI152NLin6Asym_desc-disconnectionpct.nii.gz")


def average_maps(sub_ids, path_fn):
    accum, affine, n = None, None, 0
    for sid in sub_ids:
        p = path_fn(sid)
        if not p.exists():
            continue
        img = nib.load(p)
        arr = np.asarray(img.dataobj, dtype=np.float32)
        if accum is None:
            accum = np.zeros_like(arr, dtype=np.float64)
            affine = img.affine
        accum += arr
        n += 1
    if n == 0:
        return None, None
    return accum / n, affine


def render_slnm_cluster_axials():
    assign = pd.read_csv(EXP_DIR / "umap_sLNM_subjects.csv")
    clusters = sorted([c for c in assign["cluster"].unique() if c >= 0])
    cluster_maps = {}
    for c in [*clusters, -1]:
        sub_ids = assign.loc[assign["cluster"] == c, "subject_id"].values
        arr, aff = average_maps(sub_ids, _slnm_nii_path)
        if arr is None:
            print(f"  cluster {c}: no nii files, skip")
            continue
        nib.save(nib.Nifti1Image(arr.astype(np.float32), aff),
                 CELLS_SLNM / f"cluster_{c:+d}_mean_sLNM.nii.gz")
        cluster_maps[c] = (arr, aff, len(sub_ids))
        print(f"  cluster {c}: n={len(sub_ids)}, max={arr.max():.3f}")

    pos_vals = np.concatenate([m[0][m[0] > 0] for c, m in cluster_maps.items() if c >= 0])
    vmax = float(np.percentile(pos_vals, 95))
    print(f"shared vmax = {vmax:.3f}")

    for c, (arr, aff, n_sub) in cluster_maps.items():
        if c < 0:
            continue
        img = nib.Nifti1Image(arr.astype(np.float32), aff)
        cuts = CLUSTER_Z_CUTS.get(c, N_CUTS_PER_CLUSTER)
        out_png = CELLS_SLNM / f"cluster_{c:+d}_axial.png"
        for out in (out_png, out_png.with_suffix(".svg")):
            display = plotting.plot_stat_map(
                img, display_mode="z", cut_coords=cuts, colorbar=False,
                vmax=vmax, threshold=0.005, cmap="hot", annotate=False,
                radiological=True)
            display.annotate(left_right=False, positions=True, size=12)
            display.savefig(str(out))
            display.close()
        print(f"  saved {out_png.name}")


existing = sorted(CELLS_SLNM.glob("cluster_+*_axial.png"))
if existing:
    print(f"[skip] {len(existing)} sLNM cluster maps already present")
elif not LACUNA.exists() or not any(LACUNA.glob("sub-*")):
    print("[skip] per-subject NIfTIs not found — copy them in to render cluster maps")
else:
    render_slnm_cluster_axials()

In [ ]:
# 3. Per-cluster mean maps for the lesion / fLNM views (Figs S20/S21 inputs)
VIEW_NII = {
    "lesion": "label-acuteinfarct_space-MNI152NLin6Asym_mask.nii.gz",
    "flnm":   "label-acuteinfarct_method-fnm_space-MNI152NLin6Asym_desc-tmap.nii.gz",
}
VIEW_CMAP = {
    "lesion": ("hot",      0.005, 0.95),
    "flnm":   ("cold_hot", 0.5,   0.95),
}


def render_view_cluster_axials(view):
    out_dir = (PROJECT_ROOT / "outputs" / "feature_importance" / "figures" /
               f"cells_suppl_fig3_{view}_clusters")
    out_dir.mkdir(parents=True, exist_ok=True)
    if sorted(out_dir.glob("cluster_+*_axial.png")):
        print(f"[skip] {view}: cluster maps already present")
        return
    assign = pd.read_csv(EXP_DIR / f"umap_{view}_subjects.csv")
    clusters = sorted([int(c) for c in assign["cluster"].unique() if c >= 0])
    cmap, thr, qtile = VIEW_CMAP[view]
    suffix = VIEW_NII[view]

    def path_fn(sid):
        return LACUNA / f"sub-{sid}" / "ses-01" / "anat" / f"sub-{sid}_ses-01_{suffix}"

    cluster_maps = {}
    for c in [*clusters, -1]:
        sub_ids = assign.loc[assign["cluster"] == c, "subject_id"].values
        arr, aff = average_maps(sub_ids, path_fn)
        if arr is None:
            continue
        nib.save(nib.Nifti1Image(arr.astype(np.float32), aff),
                 out_dir / f"cluster_{c:+d}_mean.nii.gz")
        cluster_maps[c] = (arr, aff, len(sub_ids))
        print(f"  [{view}] cluster {c}: n={len(sub_ids)}")

    pos_vals = np.concatenate(
        [m[0][np.abs(m[0]) > 0] for c, m in cluster_maps.items() if c >= 0])
    vmax = float(np.percentile(np.abs(pos_vals), 100 * qtile))
    for c, (arr, aff, n_sub) in cluster_maps.items():
        if c < 0:
            continue
        img = nib.Nifti1Image(arr.astype(np.float32), aff)
        out_png = out_dir / f"cluster_{c:+d}_axial.png"
        for out in (out_png, out_png.with_suffix(".svg")):
            display = plotting.plot_stat_map(
                img, display_mode="z", cut_coords=3, colorbar=False,
                vmax=vmax, threshold=thr, cmap=cmap, annotate=False,
                radiological=True)
            display.annotate(left_right=False, positions=True, size=12)
            display.savefig(str(out))
            display.close()
    print(f"  [{view}] done (vmax={vmax:.3f})")


if LACUNA.exists() and any(LACUNA.glob("sub-*")):
    for view in ("lesion", "flnm"):
        render_view_cluster_axials(view)
else:
    print("[skip] per-subject NIfTIs not found")

In [ ]:
# 3b. Lesion distribution map (Fig S1): voxelwise acute-infarct frequency
# across the analysed sample (any-domain PSCI label available), axial montage.
LESION_FREQ_NII = EXP_DIR / "lesion_frequency_map_analysed.nii.gz"
LESION_FREQ_PNG = FIG_DIR / "lesion_distribution_map_analysed.png"

if LESION_FREQ_PNG.exists():
    print(f"[skip] {LESION_FREQ_PNG.name} exists")
elif not LACUNA.exists() or not any(LACUNA.glob("sub-*")):
    print("[skip] per-subject NIfTIs not found")
else:
    import nibabel as nib
    from nilearn import plotting

    def _lesion_mask_path(sid):
        return (LACUNA / f"sub-{sid}" / "ses-01" / "anat" /
                f"sub-{sid}_ses-01_label-acuteinfarct_space-MNI152NLin6Asym_mask.nii.gz")

    clin_lm = pd.read_csv(CLIN_PATH, index_col=0)
    analysed = set(clin_lm.index[clin_lm["cognition-PSCI_present"].notna()].astype(str))
    sids = sorted(p.name[4:] for p in LACUNA.glob("sub-*")
                  if p.is_dir() and p.name[4:] in analysed)
    assert sids, "no overlap between lesion folders and analysed sample"
    acc, aff, n_used = None, None, 0
    for sid in sids:
        p = _lesion_mask_path(sid)
        if not p.exists():
            continue
        img = nib.load(p)
        arr = (np.asarray(img.dataobj) > 0).astype(np.float32)
        if acc is None:
            acc, aff = np.zeros_like(arr), img.affine
        acc += arr
        n_used += 1
    freq = acc / n_used * 100.0  # % of subjects lesioned per voxel
    nib.save(nib.Nifti1Image(freq.astype(np.float32), aff), LESION_FREQ_NII)
    print(f"lesion frequency map from {n_used} subjects; "
          f"max {freq.max():.1f}% of subjects")

    img = nib.Nifti1Image(freq.astype(np.float32), aff)
    vmax = float(np.percentile(freq[freq > 0], 99.5))
    display = plotting.plot_stat_map(
        img, display_mode="z", cut_coords=[-24, -12, 0, 12, 24, 36, 48],
        colorbar=True, vmax=vmax, threshold=0.25, cmap="hot",
        annotate=True, radiological=True, draw_cross=False)
    # (n and colour-scale unit stated in the figure caption)
    display.savefig(str(LESION_FREQ_PNG), dpi=200)
    display.savefig(str(LESION_FREQ_PNG.with_suffix(".svg")))
    display.close()
    print(f"saved {LESION_FREQ_PNG}")

In [ ]:
# 4. Acute-disconnection × WMH-burden tertile heatmaps (Fig 5b,c inputs)
from matplotlib.gridspec import GridSpec

from analysis.lib.plotting import apply_manuscript_style
apply_manuscript_style()

TARGETS_NICE = [
    ("cognition-PSCI_present",             "PSCI"),
    ("cognition-aef_domain_score",         "Attn/Exec"),
    ("cognition-ps_domain_score",          "Processing speed"),
    ("cognition-language_domain_score",    "Language"),
    ("cognition-verbalmemory_domain_score", "Verbal memory"),
    ("cognition-vsfunctions_domain_score",  "Visuosp. fn."),
    ("cognition-vsmemory_domain_score",     "Visuosp. mem."),
]

clin = pd.read_csv(CLIN_PATH, index_col=0)
wmh_snm   = load_label(LACUNA_DIR, "WMH",          "snm", "disconnectionpct")
acute_snm = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct")
common = clin.index.intersection(wmh_snm.index).intersection(acute_snm.index)
clin = clin.loc[common].copy()
clin["wmh_burden"]  = wmh_snm.loc[common].sum(axis=1)
clin["disc_burden"] = acute_snm.loc[common].sum(axis=1)
clin["disc_tertile"] = pd.qcut(clin["disc_burden"], 3, labels=["low", "med", "high"])
clin["wmh_tertile"]  = pd.qcut(clin["wmh_burden"],  3, labels=["low", "med", "high"])
print(f"N = {len(clin)} subjects with clinical + WMH + sLNM")

vmin, vmax = 0.0, 0.6


def _draw_heatmap(ax, tgt, nice, show_ylabel=False, show_xlabel=True,
                  y_label_text="Acute infarct structural\nconnectivity tertile",
                  cell_fontsize=17, n_fontsize=12, title_fontsize=16,
                  tick_fontsize=14, axislabel_fontsize=15):
    sub = clin.loc[clin[tgt].notna()]
    grid = (sub.groupby(["disc_tertile", "wmh_tertile"], observed=False)
            [tgt].agg(["mean", "size"]).reset_index())
    rate_mat = grid.pivot(index="disc_tertile", columns="wmh_tertile",
                          values="mean").reindex(["high", "med", "low"])
    n_mat = grid.pivot(index="disc_tertile", columns="wmh_tertile",
                       values="size").reindex(["high", "med", "low"])
    im = ax.imshow(rate_mat.values, cmap="RdBu_r", vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(3):
        for j in range(3):
            r = rate_mat.iloc[i, j]; n = int(n_mat.iloc[i, j])
            tc = "white" if abs(r - 0.3) > 0.20 else "#222"
            ax.text(j, i - 0.12, f"{r*100:.0f}%", ha="center", va="center",
                    fontsize=cell_fontsize, fontweight="bold", color=tc)
            ax.text(j, i + 0.24, f"n={n}", ha="center", va="center",
                    fontsize=n_fontsize, fontweight="bold", color=tc)
    ax.set_xticks(range(3))
    ax.set_xticklabels(["low", "med", "high"], fontsize=tick_fontsize, fontweight="bold")
    ax.set_yticks(range(3))
    if show_ylabel:
        ax.set_yticklabels(["high", "med", "low"], fontsize=tick_fontsize, fontweight="bold")
        ax.set_ylabel(y_label_text, fontsize=axislabel_fontsize, fontweight="bold")
    else:
        ax.set_yticklabels([])
    if show_xlabel:
        ax.set_xlabel("WMH structural connectivity tertile",
                      fontsize=axislabel_fontsize, fontweight="bold")
    ax.set_title(nice, fontsize=title_fontsize, fontweight="bold")
    return im


# PSCI-only heatmap (Fig 5b)
fig_p = plt.figure(figsize=(6.8, 8.0))
gs_p = GridSpec(2, 1, figure=fig_p, hspace=0.55,
                left=0.24, right=0.96, top=0.94, bottom=0.07,
                height_ratios=[1.0, 0.10])
ax_p = fig_p.add_subplot(gs_p[0])
im_p = _draw_heatmap(ax_p, "cognition-PSCI_present", "",
                     show_ylabel=True, show_xlabel=True,
                     cell_fontsize=22, n_fontsize=16, title_fontsize=20,
                     tick_fontsize=18, axislabel_fontsize=18)
cbax_p = fig_p.add_subplot(gs_p[1])
cb_p = fig_p.colorbar(im_p, cax=cbax_p, orientation="horizontal")
cb_p.set_label("Impairment prevalence", fontsize=18, fontweight="bold")
from matplotlib.ticker import PercentFormatter
cb_p.ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
cb_p.ax.tick_params(labelsize=16)
psci_png = FIG_DIR / "fig4_heatmap_psci.png"
fig_p.savefig(psci_png, dpi=200, bbox_inches="tight")
fig_p.savefig(psci_png.with_suffix(".pdf"), bbox_inches="tight")
fig_p.savefig(psci_png.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig_p)
print(f"saved {psci_png}")

# 6-domain heatmap grid (Fig 5c)
domain_targets = [(t, n) for t, n in TARGETS_NICE if t != "cognition-PSCI_present"]
n_dom = len(domain_targets)
fig_d = plt.figure(figsize=(3.7 * n_dom, 5.6))
gs_d = GridSpec(1, n_dom, figure=fig_d, wspace=0.18,
                left=0.05, right=0.985, top=0.90, bottom=0.22)
for ti, (tgt, nice) in enumerate(domain_targets):
    ax = fig_d.add_subplot(gs_d[ti])
    _draw_heatmap(ax, tgt, nice, show_ylabel=(ti == 0), show_xlabel=(ti == n_dom // 2))
domain_png = FIG_DIR / "fig4_heatmap_domains.png"
fig_d.savefig(domain_png, dpi=200, bbox_inches="tight")
fig_d.savefig(domain_png.with_suffix(".pdf"), bbox_inches="tight")
fig_d.savefig(domain_png.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig_d)
print(f"saved {domain_png}")

In [ ]:
# 5. Baseline covariates on the sLNM embedding (Fig S19)
from matplotlib.gridspec import GridSpecFromSubplotSpec
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

from analysis.lib.plotting import COHORT_LABELS

VIEW_B = "sLNM"
CATEGORICAL_B = ["demographics-cohort", "stroke-handedness"]
CONTINUOUS_B = ["demographics-education_STROKOG"]
EXCLUDE_B = ["demographics-ethnicity"]
PREFIXES_B = ("demographics-", "stroke-", "vascularrisk-")

NICE_B = {
    "demographics-age": "Age", "demographics-sex": "Sex",
    "demographics-ethnicity": "Ethnicity",
    "demographics-education_STROKOG": "Education",
    "demographics-cohort": "Cohort",
    "stroke-handedness": "Handedness", "stroke-NIHSS": "NIHSS",
    "stroke-prior_stroke": "Prior stroke", "stroke-prior_TIA": "Prior TIA",
    "stroke-interval_onset_NPO_days": "Onset → NPA (days)",
    "stroke-interval_onset_MRI_days": "Onset → MRI (days)",
    "stroke-total_infarct_volume_ml": "Infarct volume (ml)",
    "vascularrisk-obesity": "Obesity",
    "vascularrisk-smoking_present": "Current smoker",
    "vascularrisk-smoking_past": "Past smoker",
    "vascularrisk-hypertension": "Hypertension",
    "vascularrisk-hyperlipidemia": "Hyperlipidemia",
    "vascularrisk-diabetes_mellitus": "Diabetes",
}
BINARY_LABELS_B = {"demographics-sex": {0: "female", 1: "male"}}
DEFAULT_BINARY_B = {0: "no", 1: "yes"}
LEVEL_LABELS_B = {
    "demographics-cohort": COHORT_LABELS,
    "stroke-handedness": {"1": "left", "2": "right", "3": "ambidextrous"},
}
BIG_CATEGORICAL_B = ["demographics-cohort", "demographics-ethnicity"]


def _qual_colors(n):
    pools = ["tab10", "Dark2", "Set1", "Set2", "Accent"]
    out = []
    for name in pools:
        out.extend(plt.get_cmap(name).colors)
        if len(out) >= n:
            break
    return out[:n]


def _kind_b(series):
    s = series.dropna()
    if series.name in CONTINUOUS_B:
        return "continuous" if pd.to_numeric(s, errors="coerce").notna().any() else "categorical"
    if series.name in CATEGORICAL_B or s.dtype == object:
        return "categorical"
    return "binary" if s.nunique() == 2 else "continuous"


um = pd.read_csv(EXP_DIR / f"umap_{VIEW_B}_subjects.csv")
clin_b = pd.read_csv(CLIN_PATH, index_col=0)
clin_b.index.name = "subject_id"
cols = [c for pref in PREFIXES_B for c in clin_b.columns
        if c.startswith(pref) and c not in EXCLUDE_B]
df = um.merge(clin_b[cols].reset_index(), on="subject_id", how="left")

empty = [c for c in cols if df[c].notna().sum() == 0 or df[c].dropna().nunique() < 2]
cols = [c for c in cols if c not in empty]

big_cols   = [c for c in cols if c in BIG_CATEGORICAL_B]
small_cols = [c for c in cols if c not in BIG_CATEGORICAL_B]

n_cols = 4
n_rows = int(np.ceil(len(small_cols) / n_cols))
fig = plt.figure(figsize=(4.6 * n_cols, 3.7 * n_rows + (6.4 if big_cols else 0)))
gs = GridSpec(n_rows + (1 if big_cols else 0), n_cols, figure=fig,
              wspace=0.12, hspace=0.30,
              height_ratios=[1.0] * n_rows + ([1.75] if big_cols else []),
              left=0.035, right=0.985, top=0.945, bottom=0.035)


def _draw_continuous(ax, vals, na):
    vals = pd.to_numeric(vals, errors="coerce")
    na = vals.isna()
    v_lo, v_hi = np.nanpercentile(vals[~na].astype(float), [5, 95])
    sc = ax.scatter(df.loc[~na, "umap_x"], df.loc[~na, "umap_y"], s=7,
                    c=vals[~na].astype(float), cmap="viridis",
                    vmin=v_lo, vmax=v_hi, alpha=0.85, linewidths=0)
    cb = plt.colorbar(sc, ax=ax, fraction=0.045, pad=0.02)
    cb.ax.tick_params(labelsize=8)


def _draw_binary(ax, var, vals):
    lab_map = BINARY_LABELS_B.get(var, DEFAULT_BINARY_B)
    levels = sorted(vals.dropna().unique())
    colour = {levels[0]: "#5e8eb6", levels[-1]: "#c62828"}

    def _lab(val):
        key = int(val) if isinstance(val, (int, float, np.integer, np.floating)) else val
        return lab_map.get(key, f"{val:g}" if isinstance(
            val, (int, float, np.integer, np.floating)) else str(val))

    for val in vals.dropna().value_counts().index:
        m = vals == val
        ax.scatter(df.loc[m, "umap_x"], df.loc[m, "umap_y"], s=7,
                   c=colour[val], alpha=0.8, linewidths=0)
    handles = [Line2D([0], [0], marker="o", color="w", markersize=7,
                      markerfacecolor=colour[v], markeredgewidth=0,
                      label=_lab(v)) for v in levels]
    ax.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.0, -0.01),
              ncol=2, fontsize=8.5, frameon=False, handletextpad=0.3,
              columnspacing=1.2, borderaxespad=0.0)


def _draw_categorical(ax, var, vals, big=False, cax=None):
    levels = sorted(vals.dropna().astype(str).unique())
    lab_map = LEVEL_LABELS_B.get(var, {})

    def _level_label(lev):
        if lev in lab_map:
            return str(lab_map[lev])
        try:
            alt = str(int(float(lev)))
            if alt in lab_map:
                return str(lab_map[alt])
        except (TypeError, ValueError):
            pass
        return str(lev)

    labels = [_level_label(lev) for lev in levels]
    palette = _qual_colors(len(levels))
    codes = pd.Series(pd.Categorical(vals.astype(str),
                                     categories=levels).codes, index=vals.index)
    ok = codes >= 0
    if big:
        cmap = ListedColormap(palette)
        norm = BoundaryNorm(np.arange(len(levels) + 1) - 0.5, len(levels))
        sc = ax.scatter(df.loc[ok, "umap_x"], df.loc[ok, "umap_y"], s=11,
                        c=codes[ok], cmap=cmap, norm=norm, alpha=0.88, linewidths=0)
        cb = (fig.colorbar(sc, cax=cax, ticks=np.arange(len(levels)))
              if cax is not None else
              plt.colorbar(sc, ax=ax, ticks=np.arange(len(levels)),
                           fraction=0.055, pad=0.015))
        cb.ax.set_yticklabels(labels, fontsize=10, fontweight="bold")
        cb.ax.invert_yaxis()
        cb.outline.set_linewidth(0.5)
    else:
        for j, lev in enumerate(levels):
            m = vals.astype(str) == lev
            ax.scatter(df.loc[m, "umap_x"], df.loc[m, "umap_y"], s=7,
                       c=[palette[j]], alpha=0.85, linewidths=0, label=labels[j])
        ax.legend(loc="upper right", bbox_to_anchor=(1.0, -0.01),
                  ncol=min(3, max(1, len(levels))), fontsize=7.5, frameon=False,
                  handletextpad=0.3, columnspacing=0.9, labelspacing=0.35,
                  borderaxespad=0.0)


def _finish(ax, var, fontsize):
    ax.set_title(NICE_B.get(var, var), fontsize=fontsize, fontweight="bold")
    ax.margins(0.08)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


for i, c in enumerate(small_cols):
    ax = fig.add_subplot(gs[divmod(i, n_cols)])
    kind, vals = _kind_b(df[c]), df[c]
    na = vals.isna()
    if na.any():
        ax.scatter(df.loc[na, "umap_x"], df.loc[na, "umap_y"], s=4,
                   c=[(0.88, 0.88, 0.88, 0.45)], linewidths=0)
    if kind == "continuous":
        _draw_continuous(ax, vals, na)
    elif kind == "binary":
        _draw_binary(ax, c, vals)
    else:
        _draw_categorical(ax, c, vals, big=False)
    _finish(ax, c, 12)

for j in range(len(small_cols), n_rows * n_cols):
    ax = fig.add_subplot(gs[divmod(j, n_cols)])
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

if big_cols:
    span = gs[n_rows, 1:3] if len(big_cols) == 1 else gs[n_rows, :]
    sub_big = GridSpecFromSubplotSpec(
        1, 2 * len(big_cols), subplot_spec=span, wspace=0.06,
        width_ratios=[1.0, 0.055] * len(big_cols))
    big_axes = [fig.add_subplot(sub_big[0, 2 * k]) for k in range(len(big_cols))]
    big_caxes = [fig.add_subplot(sub_big[0, 2 * k + 1]) for k in range(len(big_cols))]
    for ax, cax, c in zip(big_axes, big_caxes, big_cols):
        vals = df[c]; na = vals.isna()
        if na.any():
            ax.scatter(df.loc[na, "umap_x"], df.loc[na, "umap_y"], s=6,
                       c=[(0.88, 0.88, 0.88, 0.45)], linewidths=0)
        _draw_categorical(ax, c, vals, big=True, cax=cax)
        ax.set_aspect("equal", adjustable="box")
        ax.set_anchor("C")
        _finish(ax, c, 15)

out_b = FIG_DIR / "umap_baseline_overlay.png"
fig.savefig(out_b, dpi=200, bbox_inches="tight")
fig.savefig(out_b.with_suffix(".svg"), bbox_inches="tight")
fig.savefig(out_b.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved {out_b}")